<a href="https://colab.research.google.com/github/bkaankaya/CineEmbed-A-Multi-Modal-Unsupervised-Film-Recommendation-System/blob/main/notebooks/00_colab_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CineEmbed — 00 Colab Setup (one-time per session)

Run this notebook ONCE at the start of each Colab session before running any training notebook.

## New setup flow

**Code** is pulled automatically from GitHub (git clone / git pull) — you no longer need to upload the repo to Drive.

**Artifacts** (feature matrices, checkpoints, results) still live on Google Drive under `MyDrive/CineEmbed/artifacts/`. Drive is the only folder you need to manage manually.

## Required Drive structure (artifacts only — set up manually once)

```
MyDrive/CineEmbed/artifacts/
├── feature_matrix.npz
├── movies_eda_final.csv
├── feature_metadata.json
├── pipeline_version.json
└── director_profile_metadata.json
```

Sub-folders `models/` and `eval/` are created automatically by the training notebooks.

In [8]:
# ─── CineEmbed — robust Colab setup (single-cell, idempotent) ─────────────────
import os, sys, json, shutil
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # 1. Mount Drive
    from google.colab import drive
    drive.mount('/content/drive')

    # 2. Resolve auth (private repo → token from Colab Secrets, else public URL)
    try:
        from google.colab import userdata
        _token = userdata.get('GITHUB_TOKEN')
        REPO_URL = f"https://{_token}@github.com/bkaankaya/CineEmbed-A-Multi-Modal-Unsupervised-Film-Recommendation-System.git"
    except Exception:
        REPO_URL = "https://github.com/bkaankaya/CineEmbed-A-Multi-Modal-Unsupervised-Film-Recommendation-System.git"

    # 3. CRITICAL: clone target MUST NOT match package name 'cineembed'
    #    (else /content + dirname='cineembed' creates a namespace-package shadow)
    REPO_ROOT = Path('/content/cineembed-repo')
    ARTIFACTS = Path('/content/drive/MyDrive/CineEmbed/artifacts')

    # 4. Clean up any prior broken state from earlier failed runs
    bad_dir = Path('/content/cineembed')   # the namespace-shadow dir
    if bad_dir.exists():
        shutil.rmtree(bad_dir, ignore_errors=True)
        print(f"[cleanup] removed namespace-shadow dir {bad_dir}")
    # Drop /content from sys.path entirely (Colab adds it; it shadows packages)
    sys.path = [p for p in sys.path if p != '/content']
    # Drop any cached broken cineembed modules
    for _m in [m for m in list(sys.modules) if m.startswith('cineembed')]:
        del sys.modules[_m]
    # Uninstall any lingering pip-registered cineembed (silent if absent)
    get_ipython().system('pip uninstall cineembed -y -q 2>/dev/null || true')

    # 5. Clone (or pull if already present)
    if not REPO_ROOT.exists():
        rc = get_ipython().system(f'git clone {REPO_URL} {REPO_ROOT}')
    else:
        get_ipython().system(f'cd {REPO_ROOT} && git pull -q')
    assert REPO_ROOT.exists() and (REPO_ROOT / 'pyproject.toml').exists(), \
        f"Clone failed — check repo URL or auth. Tried: {REPO_URL.split('@')[-1]}"

    # 6. Editable install (best-effort; sys.path is the real backstop below)
    get_ipython().system(f'pip install -e {REPO_ROOT} -q')

else:
    # Local Mac fallback
    REPO_ROOT = Path('..').resolve()
    ARTIFACTS = REPO_ROOT / 'artifacts'

# 7. Make absolutely sure the src dir is FIRST on sys.path
src_dir = str(REPO_ROOT / 'src')
sys.path = [src_dir] + [p for p in sys.path if p != src_dir]

# 8. Imports — should now succeed
import numpy as np
import torch

from cineembed import data, backbone, heads, losses, train, eval as cev

# 9. Sanity prints
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"\n✅ Setup complete")
print(f"   Repo:        {REPO_ROOT}")
print(f"   Artifacts:   {ARTIFACTS}")
print(f"   Device:      {DEVICE}")
print(f"   cineembed:   {data.__file__}")
print(f"   Drive ready: {ARTIFACTS.exists()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[cleanup] removed namespace-shadow dir /content/cineembed
Cloning into '/content/cineembed-repo'...
remote: Enumerating objects: 289, done.
remote: Counting objects: 100% (289/289), done.
remote: Compressing objects: 100% (152/152), done.
remote: Total 289 (delta 139), reused 276 (delta 126), pack-reused 0 (from 0)
Receiving objects: 100% (289/289), 217.15 KiB | 2.82 MiB/s, done.
Resolving deltas: 100% (139/139), done.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for cineembed (pyproject.toml) ... done

✅ Setup complete
   Repo:        /content/cineembed-repo
   Artifacts:   /content/drive/MyDrive/CineEmbed/artifacts
   Device:      cuda
   cineembed:   /content/cineembed-repo/sr

In [16]:
# Verify artifacts (the repo is auto-cloned above — no need to check it here)
required_artifacts = [
    ARTIFACTS / 'feature_matrix.npz',
    ARTIFACTS / 'movies_eda_final.csv',
    ARTIFACTS / 'feature_metadata.json',
    ARTIFACTS / 'pipeline_version.json',
    ARTIFACTS / 'director_profile_metadata.json',
]

all_ok = True
print('[artifacts]')
for p in required_artifacts:
    ok = p.exists()
    all_ok &= ok
    marker = '\u2713' if ok else '\u2717'
    print(f'  {marker} {p}')

# Create output sub-folders on Drive
(ARTIFACTS / 'models').mkdir(exist_ok=True)
(ARTIFACTS / 'eval').mkdir(exist_ok=True)
print(f"\nCreated/verified {ARTIFACTS / 'models'} and {ARTIFACTS / 'eval'}")

if all_ok:
    print(f"\n\u2705 Artifacts OK.")
else:
    print(f"\n\u274c Missing artifacts — upload the listed files to MyDrive/CineEmbed/artifacts/ and re-run.")

[artifacts]
  ✓ /content/drive/MyDrive/CineEmbed/artifacts/feature_matrix.npz
  ✓ /content/drive/MyDrive/CineEmbed/artifacts/movies_eda_final.csv
  ✓ /content/drive/MyDrive/CineEmbed/artifacts/feature_metadata.json
  ✓ /content/drive/MyDrive/CineEmbed/artifacts/pipeline_version.json
  ✓ /content/drive/MyDrive/CineEmbed/artifacts/director_profile_metadata.json

Created/verified /content/drive/MyDrive/CineEmbed/artifacts/models and /content/drive/MyDrive/CineEmbed/artifacts/eval

✅ Artifacts OK.


✅ Setup complete. Open 02_train_ae.ipynb next.